# 03 — Build the Schaefer-400 parcellation

Materialises `data/raw/atlas/schaefer400_fsaverage5.npz`: one parcel label per
fsaverage5 vertex, plus the parcel table (id, name, Yeo network, surface area).
Every vertex→parcel average in Phase II — equation (6) in
`src/tribe/aggregate.py` — reads this one file.

**This is not Track A.** It needs no GPU, no HuggingFace token, and no TRIBE
checkpoint, so it is *not* blocked by the pending Llama-3.2-3B access. It runs
on a free CPU runtime in a couple of minutes.

> **Runtime → Change runtime type → CPU.** A GPU here burns quota for nothing.

Run it **once**. The output is copied to Drive (Track A inference needs it in a
later session) and downloaded to your laptop (Track B analysis needs it on every
run). Do not build it independently in two places — see the last cell for why.

Nothing below computes anything itself: every cell is a bootstrap step or a call
into `scripts/build_atlas.py`.

## 1. Install

`--extra atlas` is the CPU-only slice of the dependency set: `nibabel` (reads
the FreeSurfer `.annot` files) and `nilearn` (fetches the fsaverage5 mesh), and
nothing else. The `tribe` extra also contains both, but drags in 2.5 GB of CUDA
wheels this script never touches.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
BRANCH = "phase2-tribe"
REPO_DIR = Path("/content/neurotutorsim")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# --system installs into Colab's own interpreter. `uv sync` would build a venv
# this kernel cannot import from.
subprocess.run(
    ["uv", "pip", "install", "--system", "--extra", "atlas", "-e", "."],
    cwd=REPO_DIR,
    check=True,
)
print("installed -- Runtime -> Restart session, then continue BELOW this cell")

### ⚠️ Runtime → Restart session now

`nilearn` pulls its own `scipy`/`scikit-learn` floors, and Colab has those
preloaded. Without a restart the kernel keeps the versions it imported at
startup. Everything below re-derives its own paths, so nothing from above is
needed after the restart.

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/neurotutorsim")
ATLAS_DIR = REPO_DIR / "data" / "raw" / "atlas"
ATLAS_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DRIVE = Path("/content/drive/MyDrive/NeuroTutorSim")
except ModuleNotFoundError:
    PROJECT_DRIVE = Path("./drive_local/NeuroTutorSim")   # off Colab: stay runnable
PROJECT_DRIVE.mkdir(parents=True, exist_ok=True)

import nibabel
import nilearn

print("nibabel", nibabel.__version__, "| nilearn", nilearn.__version__)
print("atlas dir:", ATLAS_DIR)

## 2. Fetch the two annotation files

`scripts/build_atlas.py` deliberately does not download these. They are the one
input whose *provenance* decides whether every downstream metric is labelled
correctly.

`fsaverage6/` and `fsaverage/` are sibling directories in the same CBIG release
holding files with **identical names**. A wrong-space annotation mislabels
cortex everywhere while looking perfectly healthy, so the path below is spelled
out in full and the payload is checked before it is written to disk.

In [ ]:
import hashlib
import urllib.request

CBIG = (
    "https://raw.githubusercontent.com/ThomasYeoLab/CBIG/master/"
    "stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/"
    "Parcellations/FreeSurfer5.3/fsaverage5/label/"
)
FILES = {h: f"{h}.Schaefer2018_400Parcels_7Networks_order.annot" for h in ("lh", "rh")}

for hemi, filename in FILES.items():
    target = ATLAS_DIR / filename
    if target.exists():
        print(f"{hemi}: already present")
    else:
        payload = urllib.request.urlopen(CBIG + filename, timeout=120).read()
        # A 404 from raw.githubusercontent arrives as an HTML page, not an error.
        if payload[:1] == b"<" or len(payload) < 50_000:
            raise SystemExit(
                f"{filename}: got {len(payload)} bytes that are not a FreeSurfer "
                f"annotation. Check the URL -- do not proceed with a partial file."
            )
        target.write_bytes(payload)
    digest = hashlib.sha256(target.read_bytes()).hexdigest()
    print(f"  {target.name}  {target.stat().st_size:,} B  sha256 {digest[:12]}")

Both files should be ~92 KB. Those 12-hex-digit prefixes are exactly what
`build_atlas.py` writes into the atlas `version` string, so you can read the
next cell's output and confirm it used these files and not something else.

*Uploading by hand instead?* Put both `.annot` files in
`/content/neurotutorsim/data/raw/atlas/` (`files.upload()` works, or drag them
into the file browser) and skip the cell above.

## 3. Build

The script asserts fsaverage5 space (10242 vertices per hemisphere) rather than
assuming it, offsets the right hemisphere into a global 1–400 id space, computes
per-parcel surface area from the pial mesh, and reads the result back with the
real `load_atlas` loader before declaring success.

Add `--force` only if you are deliberately rebuilding: it invalidates every
cached parcel table downstream.

In [ ]:
!cd /content/neurotutorsim && python scripts/build_atlas.py --config config/tribe.yaml

Expected: `400 parcels over 20484 vertices (… background), 7 networks`.

If the parcel or vertex count differs, stop — do not save the file. A wrong
count means the annotation and the mesh disagree, and the atlas is not the one
its metadata claims to be.

## 4. Save it to both places

Two destinations for two different reasons:

- **Drive** — Track A inference in a later Colab session needs the atlas, and
  Colab's own disk does not survive the session.
- **Your laptop** — Track B analysis (`src/tribe/aggregate.py`) loads it on
  every run, and that runs locally with no nilearn installed.

In [ ]:
import hashlib
import shutil

npz = ATLAS_DIR / "schaefer400_fsaverage5.npz"
if not npz.exists():
    raise SystemExit("no atlas file -- the build did not succeed")

drive_copy = PROJECT_DRIVE / npz.name
shutil.copy2(npz, drive_copy)
print(f"Drive : {drive_copy}")
print(f"size  : {npz.stat().st_size:,} B")
print(f"sha256: {hashlib.sha256(npz.read_bytes()).hexdigest()}")

try:
    from google.colab import files
    files.download(str(npz))          # -> your browser's Downloads folder
except ModuleNotFoundError:
    print("not on Colab -- skipping browser download")

## 5. On your laptop

Move the downloaded file into the repo at exactly:

```
data/raw/atlas/schaefer400_fsaverage5.npz
```

then confirm it survived the round trip — the sha256 must match the one printed
above:

```bash
python -c "from src.tribe.aggregate import load_atlas; a = load_atlas('data/raw/atlas/schaefer400_fsaverage5.npz'); print(a.n_parcels, a.version); print(a.file_hash)"
```

`data/raw/` is gitignored, so the `.npz` is **not** committed. That is intended:
its sha256 travels inside every table `aggregate.py` produces, which is what
makes the parcellation reproducible without storing it in git.

### Build once, then copy

Do not run this notebook a second time to regenerate the file on another
machine. The atlas `version` string hashes the two `.annot` files but **not**
the fsaverage5 mesh, which `nilearn` fetches at runtime. Parcel `area_mm2` — the
equation (7) network weights — is computed from that mesh, so two nilearn
versions could in principle produce different weights under an identical
`version` string. Copying the one file you built here removes the question
entirely.